# PARC2026 — T3 Track3 Inverse Data Factory

Track3向けinverse/reversed demonstrationを **Simulator-valid** に作るためのFactoryです。

禁止事項:
- forward action列を後ろから並べるだけ
- `time_reverse`, `action_reverse`, `naive_reverse` を生成sourceとして使う
- success未確認 / replay未確認のtrajectoryを学習datasetへ混ぜる

正式flow:
`define inverse task → construct inverse initial state → run simulator expert/script/policy → success check → replay validation → LeRobot 20Hz conversion → manifest registration`

M3 shortlistがまだ無くてもregistryの雛形までは作れますが、
inverse ablation 0/5/10/20% の実行はM3完了までBLOCKします。


In [ ]:
# 0/4 Mount Drive + clone repo + M3 gate
import os, json, subprocess
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
ROOT=Path("/content/parc2026"); REPO=ROOT/"py_AI"; ROOT.mkdir(parents=True, exist_ok=True)
if not (REPO/".git").exists(): subprocess.run(["git","clone","https://github.com/yu37330/py_AI.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"fetch","origin","main"],check=True); subprocess.run(["git","-C",str(REPO),"checkout","--force","origin/main"],check=True)
DRIVE=Path("/content/drive/MyDrive/parc2026-cache"); OUT=DRIVE/"track3-inverse-factory-v1"; OUT.mkdir(parents=True,exist_ok=True)
shortlist_path=DRIVE/"model-benchmark-v1/model_shortlist.json"; SHORTLIST_READY=shortlist_path.exists()
if SHORTLIST_READY:
    shortlist=json.loads(shortlist_path.read_text()); assert shortlist.get("status")=="DECIDED"; assert 1 <= len(shortlist.get("models",[])) <= 2; print("M3 shortlist:",shortlist["models"])
else:
    shortlist=None; print("M3 shortlist: MISSING — registry preparation only")


In [ ]:
# 1/4 Inventory Track3 inverse task definitions + create persistent registry
import json
from pathlib import Path
bddl_root=REPO/"compe/t3/assets/bddl_files/libero_t3"; init_root=REPO/"compe/t3/assets/init_files/libero_t3"; bddl=sorted(bddl_root.glob("*.bddl")); assert bddl,bddl_root
print("Track3 BDDL tasks:",len(bddl))
registry_path=OUT/"inverse_generation_registry.json"
registry=json.loads(registry_path.read_text()) if registry_path.exists() else {"schema_version":1,"entries":[]}
existing={e.get("inverse_task_id") for e in registry["entries"]}
for p in bddl:
    task_id=p.stem
    if task_id in existing: continue
    init_candidates=sorted(init_root.glob(task_id+"*"))
    registry["entries"].append({"forward_task_id":None,"inverse_task_id":task_id,"inverse_bddl":str(p),"inverse_initial_state":str(init_candidates[0]) if init_candidates else None,"generator_type":None,"generator_revision":None,"simulator_success":False,"replay_success":False,"converted_lerobot_20hz":False,"converted_episode_ref":None,"license_provenance":None,"notes":None})
registry_path.write_text(json.dumps(registry,indent=2)+"\n"); print("registry:",registry_path); print("entries:",len(registry["entries"])); print("=== TRACK3 REGISTRY: READY FOR GENERATION ===")


In [ ]:
# 2/4 Strict validation: reject naive reversal and unvalidated trajectories
import json
registry=json.loads((OUT/"inverse_generation_registry.json").read_text()); forbidden={"reverse","time_reverse","action_reverse","naive_reverse","reversed_actions"}; validated=[]
for e in registry["entries"]:
    generator=(e.get("generator_type") or "").lower()
    if generator in forbidden or "time_reverse" in generator or "action_reverse" in generator: raise RuntimeError(f"FORBIDDEN inverse generation source: {e['inverse_task_id']} -> {generator}")
    ref=e.get("converted_episode_ref")
    is_valid=bool(e.get("simulator_success")) and bool(e.get("replay_success")) and bool(e.get("converted_lerobot_20hz")) and bool(ref)
    if is_valid: validated.append(e)
report={"schema_version":1,"registry_entries":len(registry["entries"]),"validated_inverse_episodes":len(validated),"forbid_naive_action_reversal":True,"validation_requirements":["inverse_task_definition","inverse_initial_state","simulator_success","replay_success","lerobot_20hz_conversion"]}
(OUT/"inverse_validation_report.json").write_text(json.dumps(report,indent=2)+"\n"); print(json.dumps(report,indent=2))


In [ ]:
# 3/4 Pre-register inverse mix ratios 0/5/10/20%; do not fabricate missing demos
import json
MIX_RATIOS=[0.00,0.05,0.10,0.20]; MIX_SEED=20260906
decision_path=DRIVE/"pi05-top2-tiebreak-v1/provisional_best_dataset_recipe.json"
if not decision_path.exists(): raise RuntimeError("D10 provisional dataset decision missing")
decision=json.loads(decision_path.read_text())
if decision.get("status")!="DECIDED": raise RuntimeError("D10 still tied")
report=json.loads((OUT/"inverse_validation_report.json").read_text()); validated_count=report["validated_inverse_episodes"]
plan={"schema_version":1,"selected_dataset_variant":decision["selected_variant"],"mix_ratio_semantics":"inverse_fraction_of_mixed_training_pool","mix_seed":MIX_SEED,"ratios":[],"m3_shortlist_ready":SHORTLIST_READY}
for r in MIX_RATIOS: plan["ratios"].append({"inverse_ratio":r,"status":"READY" if (r==0.0 or validated_count>0) and SHORTLIST_READY else "BLOCKED","validated_inverse_pool_size":validated_count})
(OUT/"inverse_mix_plan.json").write_text(json.dumps(plan,indent=2)+"\n"); print(json.dumps(plan,indent=2))
if not SHORTLIST_READY: print("\nT3 ablation BLOCKED until M3 shortlist exists.")
elif validated_count==0: print("\n0% control is defined, but 5/10/20% remain BLOCKED until simulator-valid inverse demos exist.")
else: print("\nInverse mix plans are preregistered. Next step is model-specific small-scale ablation.")
